# Solution E2 · The default RandomForestRegressor is not a random forest

**Type**: Find the bug

## Objective

Find out why a colleague gets the same RMSE (0.5216) with `RandomForestRegressor` and with `BaggingRegressor(DecisionTreeRegressor())` and concludes "the random forest adds nothing to bagging": in the regressor `max_features` is **1.0** (all the features), so there is no feature subsampling and the forest is a bagging. Fix it and measure the gain.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
rmse = lambda p: mean_squared_error(y_te, p, squared=False)

## Commented solution

In [ ]:
rf_default = RandomForestRegressor(100, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
bagging = BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rf_default_rmse, bagging_rmse = rmse(rf_default.predict(X_te)), rmse(bagging.predict(X_te))
print(f"RMSE: random forest {rf_default_rmse:.4f} | bagging {bagging_rmse:.4f} -> 'the random forest adds nothing to bagging'")

**Where the bug is.** In `RandomForestClassifier` the default is `max_features="sqrt"` (at each split you choose among sqrt(p) features), in `RandomForestRegressor` it is **1.0**: all the features are considered at each split. Without the random subsampling of features a "random forest" is a
bagging of trees: same RMSE to the fourth decimal.

In [ ]:
default_max_features = RandomForestRegressor().max_features
explicit = RandomForestRegressor(100, max_features=1.0, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
same_prediction = bool(np.allclose(rf_default.predict(X_te), explicit.predict(X_te)))
print("default max_features of the regressor:", default_max_features, "| predictions identical to max_features=1.0:", same_prediction)

In [ ]:
predictions = {mf: RandomForestRegressor(100, max_features=mf, random_state=0, n_jobs=-1).fit(X_tr, y_tr).predict(X_te) for mf in [0.2, 0.33, 0.5, 0.75, 1.0]}
mf_curve = pd.Series({mf: rmse(p) for mf, p in predictions.items()})
best_mf = mf_curve.idxmin()
relative_gain = 1 - mf_curve[best_mf] / mf_curve[1.0]
prediction_corr = float(np.corrcoef(bagging.predict(X_te), predictions[best_mf])[0, 1])
print(mf_curve.round(4).to_string())
print(f"best: max_features={best_mf} | gain over the default: {relative_gain:.1%} | correlation of the predictions with the bagging: {prediction_corr:.4f}")

## Check

In [ ]:
with verify():
    check(abs(rf_default_rmse - bagging_rmse) < 0.002, "the colleague sees almost equal RMSEs (difference below 0.002)", "don't modify the colleague's code")
    check(default_max_features in (1.0, 1, "auto") and same_prediction is True, "the default is 1.0 (all the features) and the predictions match RandomForestRegressor(max_features=1.0)", "RandomForestRegressor().max_features")
    check(list(mf_curve.index) == [0.2, 0.33, 0.5, 0.75, 1.0] and best_mf in (0.33, 0.5), "the best is 0.33 or 0.5 (among the five values tried)", "idxmin of the curve")
    check(mf_curve[best_mf] < mf_curve[1.0] - 0.015 and relative_gain > 0.03, "the RMSE with the best value is over 0.015 below the default; relative gain over 3%", "1 - RMSE(best) / RMSE(1.0)")
    check(0.95 < prediction_corr < 1.0, "the predictions of the two forests are highly correlated (0.95-1.0) but not identical", "np.corrcoef of the predictions on the test set")
done()

## What you learned

- **The default `RandomForestRegressor` is a bagging:** `max_features=1.0` means all the features at each split, so the trees are correlated exactly as in bagging and the RMSE is the same (0.5216). The classifier has `"sqrt"`; the regression does not. For a real random forest you have to set it: with `0.33` the RMSE drops to 0.498 (−4.6%).
- **The gain is moderate but free:** one line of code and one parameter. With an already low test error, 4-5% less is an improvement that with other methods (tuning, ensembles) would take much more work. It is the kind of "default that is not optimal" to check first.
- **Do not trust the names:** always check the default parameter values (`estimator.get_params()`), even for classes with names like "RandomForest". They change between classification and regression and between scikit-learn versions (`"auto"` used to be all for the regressor and `sqrt` for the classifier, now they are `1.0` and `"sqrt"`).
- The predictions of the bagging and of the forest with `max_features=0.33` are highly correlated (0.99): the detail changes, not the structure of the prediction. That is why the gain is a few percentage points and not a factor.

## Common pitfall

Drawing a conclusion from a comparison between "two algorithms" with defaults that are equivalent. Before comparing two models check that the parameters really do what the name promises.

## Interview question

<details><summary>What distinguishes a random forest from a bagging of trees in scikit-learn?</summary>

Only the random subsampling of the features at each split (`max_features`). In the classifier the default is sqrt(p), so RandomForestClassifier really is a random forest; in the regressor the default is 1.0 (all the features), so without changing it it is equivalent to a bagging of trees. For regression I try values like
0.33 or 0.5 and choose them with the OOB or cross-validation.

</details>